In [ ]:
import numpy as np
import pandas as pd
import random 

class MySVM():
    def __init__(self, n_iter = 10, learning_rate = 0.001, weights = None, b = None, C = 1, sgd_sample = None, random_state = 42):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.weights = weights
        self.b = b
        self.c = C
        self.sgd_sample = sgd_sample
        self.random_state = random_state

    def fit(self, X, y, verbose = False):
        random.seed(self.random_state)
        #нужна ли конвертация?
        X_matrix = X.values
        y_vector = 2 * (y.values - 1/2) #konvert v +-1
        n_samples, count_features = X_matrix.shape
        if not self.weights:
            self.weights = np.ones(count_features)
        if not self.b:
            self.b = 1
        
        for iter in range(1, self.n_iter + 1):
            if self.sgd_sample is None:
                sample_rows_idx = range(0, X_matrix.shape[0])
            elif self.sgd_sample >= 1:
                sample_rows_idx = random.sample(range(X_matrix.shape[0]), self.sgd_sample)
            else:
                sample_rows_idx = random.sample(range(X_matrix.shape[0]), int(self.sgd_sample * X_matrix.shape[0]))

            X_batch = X_matrix[sample_rows_idx]
            y_batch = y_vector[sample_rows_idx]

            for i in range(len(sample_rows_idx)):

                if y_batch[i] * (self.weights.dot(X_batch[i]) + self.b) >= 1:
                    grad_w = 2 * self.weights
                    grad_b = 0
                else:
                    grad_w = 2 * self.weights - self.c * y_batch[i] * X_batch[i]
                    grad_b = - self.c * y_batch[i]
            
                self.weights = self.weights - self.learning_rate * grad_w
                self.b = self.b - self.learning_rate * grad_b

            loss = (np.sum(self.weights ** 2))\
                + self.c * np.mean(np.maximum(0, 1 - y_vector * (X_matrix.dot(self.weights) + self.b)))
            
            if verbose:
                if (iter % verbose == 0):
                    print(f"{iter}|loss:{loss}")

    def predict(self, X):
        X_matrix = X.values
        y_vector = np.sign(X_matrix.dot(self.weights) + self.b)
        y_vector = (y_vector + 1)//2
        return y_vector.astype(int)
        

    def get_coef(self):
        return (self.weights, self.b)

    def __str__(self):
        return (f'MySVM class: n_iter={self.n_iter}, learning_rate={self.learning_rate}')
